In [13]:
from hyperopt import hp

search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -10, 15, 1)}

In [14]:
from hyperopt import STATUS_OK

def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y
    return retval

In [15]:
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(fn = objective_func, space = search_space,
               algo = tpe.suggest, max_evals = 5,
               trials = trial_val, rstate = np.random.default_rng(seed=0))
print(best_01)

100%|██████████| 5/5 [00:00<00:00, 1753.91trial/s, best loss: -244.0]
{'x': np.float64(-4.0), 'y': np.float64(13.0)}


In [16]:
best_02 = fmin(fn = objective_func, space = search_space,
               algo = tpe.suggest, max_evals = 20,
               trials = trial_val, rstate = np.random.default_rng(seed=0))
print(best_02)

100%|██████████| 20/20 [00:00<00:00, 2154.76trial/s, best loss: -296.0]
{'x': np.float64(2.0), 'y': np.float64(15.0)}


In [17]:
print(trial_val.results)

[{'loss': -104.0, 'status': 'ok'}, {'loss': -204.0, 'status': 'ok'}, {'loss': -4.0, 'status': 'ok'}, {'loss': -244.0, 'status': 'ok'}, {'loss': 1.0, 'status': 'ok'}, {'loss': -104.0, 'status': 'ok'}, {'loss': -204.0, 'status': 'ok'}, {'loss': -4.0, 'status': 'ok'}, {'loss': -244.0, 'status': 'ok'}, {'loss': 1.0, 'status': 'ok'}, {'loss': -296.0, 'status': 'ok'}, {'loss': -60.0, 'status': 'ok'}, {'loss': 201.0, 'status': 'ok'}, {'loss': 4.0, 'status': 'ok'}, {'loss': 40.0, 'status': 'ok'}, {'loss': 0.0, 'status': 'ok'}, {'loss': -79.0, 'status': 'ok'}, {'loss': -39.0, 'status': 'ok'}, {'loss': -184.0, 'status': 'ok'}, {'loss': -19.0, 'status': 'ok'}]


In [18]:
print(trial_val.vals)

{'x': [np.float64(-6.0), np.float64(-4.0), np.float64(4.0), np.float64(-4.0), np.float64(9.0), np.float64(-6.0), np.float64(-4.0), np.float64(4.0), np.float64(-4.0), np.float64(9.0), np.float64(2.0), np.float64(10.0), np.float64(-9.0), np.float64(-8.0), np.float64(-0.0), np.float64(-0.0), np.float64(1.0), np.float64(9.0), np.float64(6.0), np.float64(9.0)], 'y': [np.float64(7.0), np.float64(11.0), np.float64(1.0), np.float64(13.0), np.float64(4.0), np.float64(7.0), np.float64(11.0), np.float64(1.0), np.float64(13.0), np.float64(4.0), np.float64(15.0), np.float64(8.0), np.float64(-6.0), np.float64(3.0), np.float64(-2.0), np.float64(0.0), np.float64(4.0), np.float64(6.0), np.float64(11.0), np.float64(5.0)]}


In [19]:
import pandas as pd

losses = [loss_dict['loss'] for loss_dict in trial_val.results]

result_df = pd.DataFrame({'x': trial_val.vals['x'],
                          'y': trial_val.vals['y'],
                          'losses': losses})

result_df

,x,y,losses
0,-6.0,7.0,-104.0
1,-4.0,11.0,-204.0
2,4.0,1.0,-4.0
3,-4.0,13.0,-244.0
4,9.0,4.0,1.0
5,-6.0,7.0,-104.0
6,-4.0,11.0,-204.0
7,4.0,1.0,-4.0
8,-4.0,13.0,-244.0
9,9.0,4.0,1.0


In [20]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()

df = pd.DataFrame(data = dataset.data, columns = dataset.feature_names)
df['target'] = dataset.target
X_features = df.iloc[:,:-1]
y_label = df.iloc[:,-1]

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X_features, y_label,
                                                    test_size=0.2, random_state=42)

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train,
                                            test_size=0.1, random_state=42)

In [22]:
xgb_search_space = {
    'max_depth': hp.quniform('max_depth', 5, 20, 1),
    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
    'learning_rate': hp.quniform('learning_rate', 0.01, 0.2, 0.01),
    'colsample_bytree': hp.quniform('colsample_bytree', 0.5, 1, 0.1)
}

In [ ]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier, 
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100,
                            max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')

    accuracy = cross_val_score(xgb_clf, X_train, y_train,
                               scoring='accuracy', cv=3)
    return {'loss': -1 * np.mean(accuracy), 'status': STATUS_OK}

In [25]:
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50,
            trials=trial_val,
            rstate=np.random.default_rng(seed=0))
print(best)

100%|██████████| 50/50 [00:06<00:00,  7.47trial/s, best loss: -0.9692256303009179]
{'colsample_bytree': np.float64(0.8), 'learning_rate': np.float64(0.08), 'max_depth': np.float64(10.0), 'min_child_weight': np.float64(2.0)}


In [26]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix, precision_recall_curve, roc_curve

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)
    f1 = f1_score(y_test, pred)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('Confusion Matrix', confusion)
    print('정확도:', accuracy, '정밀도:', precision, '재현율:', recall, 'f1 score:', f1, 'auc:', roc_auc)

In [30]:
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            early_stopping_rounds=50,
                            eval_metric='logloss',
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5))

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr,
                eval_set=evals,
                verbose=True)

pred = xgb_wrapper.predict(X_test)
pred_prob = xgb_wrapper.predict_proba(X_test)[:,1]

get_clf_eval(y_test, pred, pred_prob)

[0]	validation_0-logloss:0.59712	validation_1-logloss:0.58704
[1]	validation_0-logloss:0.54267	validation_1-logloss:0.53472
[2]	validation_0-logloss:0.49602	validation_1-logloss:0.49013
[3]	validation_0-logloss:0.45482	validation_1-logloss:0.45184
[4]	validation_0-logloss:0.41906	validation_1-logloss:0.41709
[5]	validation_0-logloss:0.38727	validation_1-logloss:0.38583
[6]	validation_0-logloss:0.35792	validation_1-logloss:0.35721
[7]	validation_0-logloss:0.33276	validation_1-logloss:0.33180
[8]	validation_0-logloss:0.30930	validation_1-logloss:0.31108
[9]	validation_0-logloss:0.28811	validation_1-logloss:0.29457
[10]	validation_0-logloss:0.26895	validation_1-logloss:0.27721
[11]	validation_0-logloss:0.25168	validation_1-logloss:0.26076
[12]	validation_0-logloss:0.23616	validation_1-logloss:0.24622
[13]	validation_0-logloss:0.22171	validation_1-logloss:0.23276
[14]	validation_0-logloss:0.20781	validation_1-logloss:0.22068
[15]	validation_0-logloss:0.19584	validation_1-logloss:0.20955
[1